In [52]:
import requests

# 基础API地址，不带查询参数
api_url = "https://115.159.159.82/api/devices"

r = requests.get(
    api_url,
    params={
        "page": 1,
        "page_size": 50
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)

data = r.json()
print("返回json所有key：", data.keys())

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
返回json所有key： dict_keys(['page', 'page_size', 'total', 'items'])


In [55]:
total = data["total"]
page_size = data["page_size"]
#向上取整算总页数
total_pages = (total + page_size - 1) // page_size

print(f"总设备数total={total}, 每页page_size={page_size},总页数total_pages={total_pages}")

devices = []

for page in range(1, total_pages + 1):
    resp = requests.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size
        },
        timeout=10
    )
    page_data = resp.json()
    page_list = page_data["items"]
    devices.extend(page_list)

print("最终采集设备数量：", len(devices))

总设备数total=5000, 每页page_size=50,总页数total_pages=100
最终采集设备数量： 5000


In [54]:
unique_dev_id = len(set(x["device_id"] for x in devices))
print("唯一 device_id 数：", unique_dev_id)

唯一 device_id 数： 5000


In [56]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

In [58]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
base_url = "https://115.159.159.82"
current_url = base_url + "/companies"
r = requests.get(current_url, timeout=10)

print(r.text[:3000])
soup = BeautifulSoup(r.text, "html.parser")

<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width,initial-scale=1">
  <meta name="description" content="SmartPark DataLab 智慧工业园区数据平台">
  <title>园区企业 · SmartPark DataLab</title>
  <link rel="icon" href="https://115.159.159.82/static/favicon.svg" type="image/svg+xml">
  <link rel="stylesheet" href="https://115.159.159.82/static/css/app.css">
  
</head>
<body>
  <a class="skip-link" href="#main">跳到主要内容</a>
  <header class="site-header">
    <div class="header-inner">
      <a class="brand" href="/" aria-label="SmartPark DataLab 首页">
        <span class="brand-mark" aria-hidden="true"><svg viewBox="0 0 32 32"><path d="M5 27V9l8-4v22M13 27V12l7-3v18M20 27V16l7-3v14M3 27h26"/></svg></span>
        <span><strong>SmartPark</strong><small>DataLab · 智慧工业园区</small></span>
      </a>
      <button class="nav-toggle" type="button" aria-expanded="false" aria-controls="main-nav">菜单</button>
      <nav id="main-nav" class="main-nav"

In [60]:
cards = soup.select("article.company-card")
print("本页企业条数", len(cards))

本页企业条数 20


In [15]:
card = cards[0]

company_id = card.get("data-company-id")
print("company_id:", company_id)

name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".badge").get_text(strip=True)
print("企业名称：", name, "状态：", status)

details = {}
for item in card.select("dl > div"):
    k = item.select_one("dt").get_text(strip=True)
    v = item.select_one("dd").get_text(strip=True)
    details[k] = v
print(details)

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台","").strip())
print(industry, zone, device_count)

company_id: C000001
企业名称： 临港精工001有限公司 状态： 运营中
{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}
汽车零部件 智造一区 6


In [27]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
base_url = "https://115.159.159.82"
current_url = base_url + "/companies"
r = requests.get(current_url, timeout=10)
# !!!打印前3000字符原始网页源代码，直接看原生HTML，摆脱浏览器翻译坑
print(r.text[:3000])
soup = BeautifulSoup(r.text, "html.parser")

<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width,initial-scale=1">
  <meta name="description" content="SmartPark DataLab 智慧工业园区数据平台">
  <title>园区企业 · SmartPark DataLab</title>
  <link rel="icon" href="https://115.159.159.82/static/favicon.svg" type="image/svg+xml">
  <link rel="stylesheet" href="https://115.159.159.82/static/css/app.css">
  
</head>
<body>
  <a class="skip-link" href="#main">跳到主要内容</a>
  <header class="site-header">
    <div class="header-inner">
      <a class="brand" href="/" aria-label="SmartPark DataLab 首页">
        <span class="brand-mark" aria-hidden="true"><svg viewBox="0 0 32 32"><path d="M5 27V9l8-4v22M13 27V12l7-3v18M20 27V16l7-3v14M3 27h26"/></svg></span>
        <span><strong>SmartPark</strong><small>DataLab · 智慧工业园区</small></span>
      </a>
      <button class="nav-toggle" type="button" aria-expanded="false" aria-controls="main-nav">菜单</button>
      <nav id="main-nav" class="main-nav"

In [40]:
companies = []
current_url = base_url + "/companies"

for _ in range(3):
    print("正在请求页面：", current_url)
    resp = requests.get(current_url, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")
    
    cards = soup.select("article.company-card")
    
    for card in cards:
        cid = card.get("data-company-id")
        name = card.select_one("h2 a").get_text(strip=True)
        status = card.select_one(".badge").get_text(strip=True)
        
        details = {}
        for item in card.select("dl > div"):
            k = item.select_one("dt").get_text(strip=True)
            v = item.select_one("dd").get_text(strip=True)
            details[k]=v
        
        one_company = {
            "company_id": cid,
            "name": name,
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].replace("台","").strip()),
            "status": status
        }
        companies.append(one_company)
    
    next_link = None
    pagination_div = soup.select_one(".pagination")
    if pagination_div:
        for a in pagination_div.select("a"):
            txt = a.get_text(strip=True)
            if txt == "下一页":
                next_link = a
                break
    
    if not next_link:
        print("没有下一页，提前结束循环")
        break
    
    rel_href = next_link.get("href")
    current_url = urljoin(current_url, rel_href)

print("采集到企业总数量：", len(companies))

正在请求页面： https://115.159.159.82/companies
正在请求页面： https://115.159.159.82/companies?page=2
正在请求页面： https://115.159.159.82/companies?page=3
采集到企业总数量： 60


In [69]:
from collections import Counter

print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))

print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0
for comp in companies:
    cid = comp["company_id"]
    expected = comp["device_count"]
    actual = device_counts.get(cid, 0)
    if expected != actual:
        mismatch_count +=1

print("不一致企业数 =", mismatch_count)

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000
不一致企业数 = 0


In [70]:
import json
import os

os.makedirs("data", exist_ok=True)

with open("data/companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

with open("data/devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)
